# SMO — T4 Memory Benchmark (Colab/Kaggle)

Compara **AdamW-fp32 vs bitsandbytes AdamW8bit vs SMO vs SMO-8bit** en una sola GPU de 16 GB (T4).

| Celda | Qué hace | Tiempo aprox. |
|---|---|---|
| Setup | clona repo, instala bitsandbytes | ~1 min |
| Calidad GPT | char-GPT ~40M, 1000 steps, tabla comparativa | ~15 min |
| Calidad ViT | TinyViT CIFAR-10, 3 epochs | ~30–45 min |
| Killer demo | modelo ~700M+: espera `AdamW=OOM`, `SMO-8bit=ok` | ~10–20 min |

Resultados en `benchmarks/results/t4_*_memory_results.json` (la última celda los tabula).

> Notas de equidad: weight_decay=0 en todos, mismo schedule coseno+warmup, clip y seed.
> `--amp` usa autocast fp16 solo en fwd/bwd; los estados del optimizador siguen en fp32.


In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory / 1e9:.1f} GB)")
assert torch.cuda.is_available(), 'Activa GPU: Entorno de ejecucion > Cambiar tipo de entorno > T4'

In [ ]:
import os
BASE = "/content" if os.path.isdir("/content") else "/kaggle/working"

REPO_URL = "https://github.com/mcarbonell/supermario_optimizer.git"

%cd {BASE}
!rm -rf smo_optimizer
!git clone --depth 1 {REPO_URL} smo_optimizer
%cd smo_optimizer

%pip install -q bitsandbytes

!python -c "from smo import SMO, SMO8bit; import benchmarks; print('SMO import OK')"

In [ ]:
# ---- 1) Calidad: GPT ~40M, 1000 steps (~15 min) ----
!python -m benchmarks.suites.comparison.t4_memory_benchmark --suite gpt --steps 1000 --amp --seed 1234

In [ ]:
# ---- 2) Calidad: TinyViT CIFAR-10, 3 epochs (~30-45 min) ----
!python -m benchmarks.suites.comparison.t4_memory_benchmark --suite vit --epochs 3 --amp --seed 1234

## Killer demo: ¿entrena donde AdamW no cabe?

Modelo grande (~700M+ params). Con fp32, AdamW necesita ~12 bytes/param solo en
pesos+grads+estado → debería quedarse sin memoria en 16 GB. SMO-8bit comprime el estado
a ~1 byte/param → debería completar el run. El script registra el OOM por optimizador
sin abortar, así que la propia tabla demuestra el punto.

Si AdamW *no* llega a OOM, sube `--d_model` o `--block_size`; si ambos hacen OOM,
baja `--batch`.


In [ ]:
# ---- 3) Killer demo (~10-20 min) ----
!python -m benchmarks.suites.comparison.t4_memory_benchmark --suite gpt \
    --d_model 1536 --layers 32 --block_size 384 --batch 8 --steps 300 \
    --amp --eval_interval 50 --seed 1234

In [ ]:
# ---- Resumen de todos los resultados ----
import glob
import json

import pandas as pd

rows = []
for path in sorted(glob.glob("benchmarks/results/t4_*_memory_results*.json")):
    bundle = json.load(open(path))
    for r in bundle["runs"]:
        m = r["metrics"]
        rows.append({
            "suite": bundle["summary"]["suite"],
            "optimizer": r["variant"],
            "status": m.get("status", "?"),
            "peak_alloc_MB": m.get("_peak_alloc_mb"),
            "reserved_MB": m.get("peak_reserved_mb"),
            "state_MB": m.get("persistent_state_mb"),
            "metric": m.get(r.get("metric_key", "")),
            "coverage_pct": m.get("coverage_pct"),
            "wall_s": m.get("wall_s"),
        })

df = pd.DataFrame(rows)
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(df)

In [ ]:
# ---- Descargar resultados (Colab) ----
import shutil

shutil.make_archive("t4_results", "zip", "benchmarks/results")
try:
    from google.colab import files

    files.download("t4_results.zip")
except ImportError:
    print("En Kaggle: mira /kaggle/working/t4_results.zip en el panel de Output")